In [1]:
import pandas as pd
import numpy as np
import matplotlib as mlb
from scipy.stats import chi2_contingency
from dotenv import load_dotenv
import os
load_dotenv()

True

IMPORT THE DATA

In [3]:
sales = pd.read_csv(os.getenv("CLEAN_DATA_PATH") + "/sales_train_cnt_0.csv")

PREP WORK


In [4]:
cat_cols = ["shop_id", "item_id", "item_category_id", "month_block_num"]
numeric_cols = ["item_price", "item_cnt_month"]

In [5]:
def cramer_v(col1: pd.Series, col2:pd.Series) -> float:
    """
    Returns the Cramer V correlation between two nominal variables
    """
    # Create a contingency table
    contingency_table = pd.crosstab(col1, col2)
    chi2_statistic, p_value, dof, expected = chi2_contingency(contingency_table)
    
    # Calculate Cramer's V
    n = contingency_table.sum().sum()
    phi2 = chi2_statistic / n
    r, k = contingency_table.shape
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    k_corr = k - (k - 1) * (k - 2) / (n - 1)
    r_corr = r - (r - 1) * (r - 2) / (n - 1)
    v = np.sqrt(phi2corr / min(k_corr - 1, r_corr - 1))
    
    return v
    return 2.0

In [ ]:
def get_nominal_var_corr_matrix() -> pd.DataFrame:
    """
    Returns the Cramer V correlation matrix of all nominal variables
    in sales dataframe
    """
    # empty correlation matrix dataframe
    corr_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols)
    num_cat_vars = len(cat_cols)
    corr_values = {}
    for k in range(num_cat_vars):
        corr_values[k] = []

    for i in range(num_cat_vars):
        for j in range(i, num_cat_vars):
            vari, varj = [cat_cols[i], cat_cols[j]]
            # matrix diagonal case (in correlation matrix)
            # m at row i and column i is ~1.00
            if (i == j):
                corr_matrix.loc[vari, varj] = cramer_v(sales[vari], sales[varj])
            # we haven't calculated corr(vari, varj)
            elif (i < j):
                corr_coef = cramer_v(sales[vari], sales[varj])
                #store correlation  in dict
                corr_matrix.loc[vari, varj] = corr_coef
                corr_matrix.loc[varj, vari] = corr_coef
            
            
    
    return corr_matrix

In [7]:

def get_cat_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Turns the the values of columns that are categorical variables 
    from integer to string
    """
    new_df = df[numeric_cols]
    for column in cat_cols:
        new_df[column] = df[column].astype(str)
    return new_df

In [8]:
sales = get_cat_columns(sales)

Basic Information

In [9]:
sales.shape

(2935843, 6)

In [40]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 2935843 entries, 0 to 2935842
Data columns (total 6 columns):
 #   Column            Dtype  
---  ------            -----  
 0   item_price        float64
 1   item_cnt_month    float64
 2   shop_id           str    
 3   item_id           str    
 4   item_category_id  str    
 5   month_block_num   str    
dtypes: float64(2), str(4)
memory usage: 134.4 MB


SUMMARY STATISTICS

In [10]:
print("Summary for categorical variables")
sales[cat_cols].describe()

Summary for categorical variables


,shop_id,item_id,item_category_id,month_block_num
count,2935843,2935843,2935843,2935843
unique,60,21807,84,34
top,31,20949,40,11
freq,235636,31340,564651,143246


In [38]:
print("SUmmary for numerical variables")
sales[numeric_cols].describe()

SUmmary for numerical variables


,item_price,item_cnt_month
count,2.935843e+06,2.935843e+06
mean,8.908535e+02,7.410409e+00
std,1.729801e+03,3.033649e+01
min,7.000000e-02,0.000000e+00
25%,2.490000e+02,1.000000e+00
50%,3.990000e+02,2.000000e+00
75%,9.990000e+02,5.000000e+00
max,3.079800e+05,2.253000e+03


In [41]:
sales.corr(numeric_only=True)

,item_price,item_cnt_month
item_price,1.000000,0.002631
item_cnt_month,0.002631,1.000000


In [11]:
get_nominal_var_corr_matrix()

,shop_id,item_id,item_category_id,month_block_num
shop_id,1.0,0.174125,0.150569,0.082651
item_id,0.174125,1.0,0.996293,0.301734
item_category_id,0.150569,0.996293,1.0,0.075616
month_block_num,0.082651,0.301734,0.075616,1.0


In [61]:
corr_matrix2 = pd.DataFrame(index=cat_cols, columns=cat_cols)

for var1 in cat_cols:
    for var2 in cat_cols:
        
        var_combination = var1 + var2
        # matrix diagonal case (in correlation matrix)
        # m at row i and column i is ~1.00
        corr_coef = cramer_v(sales[var1], sales[var2])
            
        corr_matrix2.loc[var1, var2] = corr_coef

In [62]:
corr_matrix2

,shop_id,item_id,item_category_id,month_block_num
shop_id,1.0,0.174125,0.150569,0.082651
item_id,0.174125,1.0,0.996293,0.301734
item_category_id,0.150569,0.996293,1.0,0.075616
month_block_num,0.082651,0.301734,0.075616,1.0


In [58]:
cramer_v(sales["shop_id"], sales["item_id"])

np.float64(0.17412515193659883)